# 08. Ruído local e fundo do papel

Este notebook avalia duas técnicas de PDI baseadas no **papel** e não no texto, primeiro como mapas de região e depois, se passarem, como features de um classificador candidato.

- **Técnica A, ruído local.** Em cada bloco de fundo, mede-se o desvio do resíduo da imagem (imagem menos a mediana 3 x 3, slides 201-216 do material da disciplina). Um trecho colado ou apagado costuma ter ruído de digitalização diferente do resto da página.
- **Técnica B, fundo do papel.** Ajusta-se uma superfície polinomial de grau 2 (mínimos quadrados) aos pixels de fundo, que modela a iluminação lenta do scanner. O que sobra por bloco mede a rugosidade e o brilho do papel; um trecho editado tende a destoar.

Nas duas, o mapa é um z robusto (mediana e MAD da própria imagem) por bloco de lado B, com B em {32, 64}. Só pixels de fundo contam: a máscara de Otsu do texto, dilatada, fica de fora (Otsu e a dilatação são métodos de apoio; não aparecem nos slides). As features de imagem são o percentil 99 de |z| e a fração de blocos com |z| > 3.

A grade, os critérios e os hashes do código foram gravados em `results/grade_ruido_fundo.json` antes de qualquer execução sobre imagens reais. O split `test` do *Find it again!* não é aberto. Seed 42. Os números aparecem apenas como saída das células; este notebook não registra conclusões.

In [1]:
import sys
sys.path.append("../src")
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from classification import bootstrap_auc, cv_evaluate
from comparacao_cv import auc_por_particao, diferenca_pareada
from dataset_loader import load_dataset, load_gray
from features import FEATURE_NAMES, load_or_extract
from features_v2 import V2_NAMES, load_or_extract_v2
from evaluation import wilson_score_interval
from ruido_fundo import NOMES, TAMANHOS, auc_regioes, features_lote, hit_rate_evaluation, mapas_imagem

RAIZ = Path("..")
DATASET_ROOT = str(RAIZ / "data/findit2")

## 1. Grade e critérios congelados

A célula confere que os arquivos de código atuais têm os hashes registrados e mostra a definição das duas técnicas e os critérios da etapa de região.

In [2]:
grade = json.loads((RAIZ / "results/grade_ruido_fundo.json").read_text(encoding="utf-8"))
print("congelado em:", grade["congelado_em"])
for arquivo, esperado in grade["codigo"].items():
    print(arquivo, "idêntico ao congelado:", hashlib.sha256((RAIZ / arquivo).read_bytes()).hexdigest() == esperado)
print(json.dumps(grade["etapa_1_regiao"], ensure_ascii=False, indent=1))

congelado em: 2026-11-06T11:30:23
src/ruido_fundo.py idêntico ao congelado: True
src/comparacao_cv.py idêntico ao congelado: True
{
 "conjunto": "train do Find it again!, imagens adulteradas com regiões anotadas",
 "regiao_real": "maior |z| entre os blocos válidos que tocam as regiões modificadas (is_original_area is not True)",
 "acaso": "generate_pseudo_region (mesmo tamanho da maior região modificada, com texto, sem sobreposição com as anotadas), 20 sorteios por imagem, seed 42, mínimo de 10 válidos, como evaluate_region_technique",
 "auc": "por imagem, fração de pseudo-regiões com score menor que o da região real (empate = 1/2); AUC final = média entre imagens",
 "ic": "bootstrap sobre as imagens, 2000 reamostragens, seed 42, percentis 2,5 e 97,5",
 "tecnica_passa_se": "limite inferior do IC > 0,5 para algum B",
 "B_vencedor": "o B de maior AUC entre os que passam (empate: menor B)",
 "complemento": "acerto contra o acaso por evaluate_region_technique, reportado, fora do critério"


## 2. Etapa 1: mapas no train e AUC no nível de região

Calculam-se os quatro mapas (duas técnicas x dois valores de B) de todas as imagens do split `train`. Contam-se os mapas indefinidos (menos de 10 blocos válidos, ou MAD zero).

O critério de região mede se o mapa destaca a região adulterada acima de pseudo-regiões sorteadas ao acaso. Em cada imagem adulterada, a região real vale o maior |z| entre os blocos que a tocam; as pseudo-regiões têm o tamanho da maior região modificada, contêm texto e não se sobrepõem às regiões anotadas (as mesmas regras de `evaluate_region_technique`, 20 sorteios por imagem, seed 42). A AUC da imagem é a fração de pseudo-regiões com score menor que o da região real, e a AUC final é a média entre imagens, com IC de bootstrap (2000 reamostragens sobre as imagens). Uma AUC de 0,5 é o acaso. Como complemento, reporta-se o acerto real contra o acaso calculado por `evaluate_region_technique`.

In [3]:
train = [s for s in load_dataset(DATASET_ROOT, "train") if s.image_path.exists()]
# em série: o ajuste do polinômio (LAPACK) trava dentro de processos criados por fork no macOS, e o cálculo é rápido
mapas = {str(s.image_path): mapas_imagem(load_gray(s.image_path)) for s in train}
indefinidos = {chave: sum(m[chave] is None for m in mapas.values()) for chave in next(iter(mapas.values()))}
print("imagens do train:", len(train), "| adulteradas com regiões:", sum(s.is_tampered and bool(s.regions) for s in train))
print("mapas indefinidos por configuração:", indefinidos)

aviso: 1 imagens com anotação ausente/malformada tratadas como regions=[] (train)
aviso: 2 imagens com forged divergente da anotação (train): ['X51008114260.png', 'X51006619709.png']
aviso: 2 regiões marcadas como área original mas com modificação real (train)


imagens do train: 577 | adulteradas com regiões: 93
mapas indefinidos por configuração: {'local_32': 577, 'fundo_32': 0, 'local_64': 577, 'fundo_64': 0}


In [4]:
linhas = []
for nome in NOMES:
    for b in TAMANHOS:
        r = auc_regioes(train, mapas, nome, b)
        h = hit_rate_evaluation([s for s in train if s.is_tampered and s.regions], mapas, nome, b)
        linhas.append({**{k: r[k] for k in ("tecnica", "B", "n_imagens", "auc", "ic_min", "ic_max")}, "exclusoes": r["exclusoes"],
                       "acerto_real": h["real_hit_rate"], "acerto_acaso": h["chance_hit_rate"], "n_evaluate": h["n_samples"]})
tabela = pd.DataFrame(linhas)
pd.set_option("display.width", 250)
print(tabela.drop(columns=["exclusoes"]).round(3).to_string(index=False))
print(tabela[["tecnica", "B", "exclusoes"]].to_string(index=False))

tecnica  B  n_imagens   auc  ic_min  ic_max  acerto_real  acerto_acaso  n_evaluate
  local 32          0   NaN     NaN     NaN        0.000         0.000           0
  local 64          0   NaN     NaN     NaN        0.000         0.000           0
  fundo 32         89 0.616   0.549   0.680        0.544         0.489          79
  fundo 64         88 0.632   0.568   0.692        0.624         0.506          85
tecnica  B                                                                       exclusoes
  local 32 {'mapa_indefinido': 93, 'regiao_real_sem_bloco': 0, 'poucas_pseudo_validas': 0}
  local 64 {'mapa_indefinido': 93, 'regiao_real_sem_bloco': 0, 'poucas_pseudo_validas': 0}
  fundo 32  {'mapa_indefinido': 0, 'regiao_real_sem_bloco': 2, 'poucas_pseudo_validas': 2}
  fundo 64  {'mapa_indefinido': 0, 'regiao_real_sem_bloco': 2, 'poucas_pseudo_validas': 3}


Critério congelado: a técnica passa se o limite inferior do IC da AUC for maior que 0,5 para algum B. O B vencedor é o de maior AUC entre os que passam (empate: o menor).

In [5]:
aprovadas = {}
for nome in NOMES:
    passam = tabela[(tabela["tecnica"] == nome) & (tabela["ic_min"] > 0.5)].sort_values(["auc", "B"], ascending=[False, True])
    aprovadas[nome] = int(passam.iloc[0]["B"]) if len(passam) else None
    print(nome, "-> passa:", aprovadas[nome] is not None, "| B vencedor:", aprovadas[nome])
parar = all(b is None for b in aprovadas.values())
print("nenhuma técnica passou (parar):", parar)

local -> passa: False | B vencedor: None
fundo -> passa: True | B vencedor: 64
nenhuma técnica passou (parar): False


Registro da etapa de região em `results/diagnostico_ruido_fundo_train.json`.

In [6]:
saida = {"escopo": "só train do Find it again!", "mapas_indefinidos": indefinidos,
         "regiao": tabela.to_dict("records"), "aprovadas_B_vencedor": aprovadas, "parar": bool(parar),
         "codigo_sha256": grade["codigo"]}
(RAIZ / "results/diagnostico_ruido_fundo_train.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("gravado")

gravado


## 3. Etapa 2: classificador candidato

Pelo critério da etapa de região, só a técnica B (fundo do papel) passou, com B = 64 vencedor (a técnica A não gerou mapa em nenhuma imagem do train; ver a tabela da seção 2). O candidato usa as 12 features do classificador v2 mais as duas features de imagem da técnica B com B = 64 (percentil 99 de |z| e fração de blocos com |z| > 3).

Os dados são os mesmos do v2: train + val do *Find it again!* (o `test` não é aberto). As features novas são calculadas nas imagens de train e val e guardadas em `data/features_ruido_train.csv` e `data/features_ruido_val.csv`. Imagens com mapa indefinido (se houver) recebem a mediana das imagens definidas, e a contagem é reportada.

In [7]:
amostras = {split: load_dataset(DATASET_ROOT, split) for split in ("train", "val")}
mapas_val = {str(s.image_path): mapas_imagem(load_gray(s.image_path)) for s in amostras["val"]}
todos_mapas = {**mapas, **mapas_val}
tabelas = {}
for split, S in amostras.items():
    novas = pd.DataFrame.from_dict(features_lote({str(s.image_path): todos_mapas[str(s.image_path)] for s in S}), orient="index")
    novas.insert(0, "image", [Path(caminho).name for caminho in novas.index])
    novas = novas.reset_index(drop=True)
    novas.to_csv(RAIZ / f"data/features_ruido_{split}.csv", index=False)
    v1 = load_or_extract(S, str(RAIZ / f"data/features_{split}.csv"))
    v2 = load_or_extract_v2(S, str(RAIZ / f"data/features_v2_{split}.csv"))
    tabelas[split] = v1.merge(v2, on="image", validate="one_to_one").merge(novas, on="image", validate="one_to_one")
dados = pd.concat([tabelas["train"], tabelas["val"]], ignore_index=True)
NOVAS = ["fundo_64_p99", "fundo_64_frac"]
indefinidas = int(dados[NOVAS].isna().any(axis=1).sum())
dados[NOVAS] = dados[NOVAS].fillna(dados[NOVAS].median())
y = dados["is_tampered"].to_numpy()
print("train + val:", len(dados), "| adulteradas:", int(y.sum()), "| autênticas:", int((y == 0).sum()))
print("imagens com mapa de fundo indefinido (features imputadas):", indefinidas)
print(dados.groupby("is_tampered")[NOVAS].mean().rename(index={0: "autêntica", 1: "adulterada"}).round(3).to_string())

aviso: 1 imagens com anotação ausente/malformada tratadas como regions=[] (train)
aviso: 2 imagens com forged divergente da anotação (train): ['X51008114260.png', 'X51006619709.png']
aviso: 2 regiões marcadas como área original mas com modificação real (train)
aviso: 1 imagens listadas em val.txt e ausentes de val/ foram ignoradas: ['X51006619709.png']
aviso: 1 regiões marcadas como área original mas com modificação real (val)


train + val: 769 | adulteradas: 127 | autênticas: 642
imagens com mapa de fundo indefinido (features imputadas): 0
             fundo_64_p99  fundo_64_frac
is_tampered                             
autêntica           9.146          0.053
adulterada          9.133          0.052


Reprodução do protocolo do v2. A validação cruzada é estratificada, 5 folds x 3 repetições, seed 42, com padronização e regressão logística balanceada (`class_weight="balanced"`, `max_iter=1000`). Como `cv_evaluate` devolve só médias, `auc_por_particao` repete o mesmo divisor e a mesma semente para obter a AUC de cada partição. A célula confere que a média reproduz a AUC externa registrada em `results/configuracao_congelada.json` (0,665416 com desvio 0,046784).

In [8]:
modelo = lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
colunas_v2 = FEATURE_NAMES + V2_NAMES
colunas_candidato = colunas_v2 + NOVAS
config = json.loads((RAIZ / "results/configuracao_congelada.json").read_text(encoding="utf-8"))
auc_v2 = auc_por_particao(modelo, dados[colunas_v2].to_numpy(), y)
resumo_v2 = cv_evaluate(modelo, dados[colunas_v2].to_numpy(), y)
print("v2: média das partições = %.6f | desvio = %.6f | cv_evaluate: %.6f +- %.6f | registrado: %.6f +- %.6f" % (
    auc_v2.mean(), auc_v2.std(), resumo_v2["auc_mean"], resumo_v2["auc_std"],
    config["auc_externa_cv"]["media"], config["auc_externa_cv"]["desvio"]))
print("reproduz o registrado:", abs(auc_v2.mean() - config["auc_externa_cv"]["media"]) < 1e-9)

v2: média das partições = 0.665416 | desvio = 0.046784 | cv_evaluate: 0.665416 +- 0.046784 | registrado: 0.665416 +- 0.046784
reproduz o registrado: True


Comparação lado a lado, nas mesmas 15 partições. A diferença pareada é AUC(candidato) menos AUC(v2) em cada partição. O IC 95% usa a correção de Nadeau e Bengio para validação cruzada repetida (as partições compartilham dados, e o IC ingênuo seria estreito demais); o IC ingênuo é mostrado só para comparação.

In [9]:
auc_cand = auc_por_particao(modelo, dados[colunas_candidato].to_numpy(), y)
comparacao = diferenca_pareada(auc_cand, auc_v2)
por_particao = pd.DataFrame({"v2": auc_v2, "candidato": auc_cand, "diferenca": auc_cand - auc_v2})
print(por_particao.round(4).to_string())
print("v2:        AUC = %.4f +- %.4f" % (auc_v2.mean(), auc_v2.std()))
print("candidato: AUC = %.4f +- %.4f" % (auc_cand.mean(), auc_cand.std()))
print("diferença pareada média = %.4f (desvio entre partições %.4f)" % (comparacao["diferenca_media"], comparacao["desvio"]))
print("IC 95%% corrigido (Nadeau e Bengio): [%.4f, %.4f]" % tuple(comparacao["ic95_corrigido"]))
print("IC 95%% ingênuo: [%.4f, %.4f]" % tuple(comparacao["ic95_ingenuo"]))
print("partições em que o candidato supera o v2:", int((auc_cand > auc_v2).sum()), "de", len(auc_v2))

        v2  candidato  diferenca
0   0.6412     0.6271    -0.0141
1   0.6983     0.6983     0.0000
2   0.6195     0.5984    -0.0211
3   0.6695     0.6608    -0.0087
4   0.6909     0.6928     0.0019
5   0.7374     0.7142    -0.0231
6   0.6581     0.6590     0.0009
7   0.6028     0.6047     0.0019
8   0.7681     0.7513    -0.0167
9   0.5925     0.5859    -0.0066
10  0.6623     0.6653     0.0030
11  0.7058     0.6974    -0.0084
12  0.6623     0.6636     0.0012
13  0.6322     0.6028    -0.0295
14  0.6403     0.6406     0.0003
v2:        AUC = 0.6654 +- 0.0468
candidato: AUC = 0.6575 +- 0.0464
diferença pareada média = -0.0079 (desvio entre partições 0.0107)
IC 95% corrigido (Nadeau e Bengio): [-0.0208, 0.0049]
IC 95% ingênuo: [-0.0138, -0.0020]
partições em que o candidato supera o v2: 6 de 15


AUC isolada de cada feature nova (adulterada contra autêntica, nas 769 imagens de train e val), com IC de bootstrap sobre as imagens (2000 reamostragens, seed 42). Uma AUC de 0,5 é o acaso.

In [10]:
isoladas = {}
for nome in NOVAS:
    valor = dados[nome].to_numpy()
    isoladas[nome] = {"auc": float(roc_auc_score(y, valor)), "ic_bootstrap": list(bootstrap_auc(y, valor))}
    print(nome, "AUC = %.3f, IC bootstrap [%.3f, %.3f]" % (isoladas[nome]["auc"], *isoladas[nome]["ic_bootstrap"]))

fundo_64_p99 AUC = 0.498, IC bootstrap [0.448, 0.552]


fundo_64_frac AUC = 0.497, IC bootstrap [0.444, 0.549]


Critério congelado: o candidato vence se a AUC externa média na validação cruzada for pelo menos 0,70 **e** o limite inferior do IC 95% corrigido da diferença pareada contra o v2 for maior que zero.

In [11]:
atingiu_auc = bool(auc_cand.mean() >= 0.70)
atingiu_diferenca = bool(comparacao["ic95_corrigido"][0] > 0)
print("AUC externa do candidato >= 0,70:", atingiu_auc, "(%.4f)" % auc_cand.mean())
print("limite inferior do IC95 corrigido da diferença > 0:", atingiu_diferenca, "(%.4f)" % comparacao["ic95_corrigido"][0])
print("critério atingido:", atingiu_auc and atingiu_diferenca)
saida = {"escopo": "train + val do Find it again!; test não aberto", "n_imagens": int(len(dados)), "features_novas": NOVAS,
         "tecnicas_aprovadas": aprovadas, "imagens_com_mapa_indefinido": indefinidas,
         "v2": {"auc_media": float(auc_v2.mean()), "auc_desvio": float(auc_v2.std()), "auc_por_particao": [float(v) for v in auc_v2],
                "reproduz_registrado": bool(abs(auc_v2.mean() - config["auc_externa_cv"]["media"]) < 1e-9)},
         "candidato": {"auc_media": float(auc_cand.mean()), "auc_desvio": float(auc_cand.std()), "auc_por_particao": [float(v) for v in auc_cand]},
         "diferenca_pareada": comparacao, "auc_isolada": isoladas,
         "criterio": {"auc_externa_minima": 0.70, "atingiu_auc": atingiu_auc, "atingiu_diferenca": atingiu_diferenca,
                      "atingido": atingiu_auc and atingiu_diferenca},
         "codigo_sha256": grade["codigo"]}
(RAIZ / "results/classificador_ruido_fundo_cv.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("gravado")

AUC externa do candidato >= 0,70: False (0.6575)
limite inferior do IC95 corrigido da diferença > 0: False (-0.0208)
critério atingido: False
gravado
